<a href="https://colab.research.google.com/github/loudamn78-gif/thakaa-flow-training/blob/main/week2/day12/week2_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
raw = pd.read_csv("loan.csv")

In [11]:
df = raw.copy()

# 1. الأعمدة التصنيفية: mode
for col in ["Gender", "Married", "Self_Employed"]:
    print(col, "| الناقص:", df[col].isna().sum(), "| الأكثر تكرارًا:", df[col].mode()[0])
    df[col] = df[col].fillna(df[col].mode()[0])

# 2. Dependents: mode ثم تحويل 3+ إلى 3
print("Dependents قبل:", df["Dependents"].value_counts(dropna=False).to_dict())
df["Dependents"] = df["Dependents"].fillna(df["Dependents"].mode()[0])
df["Dependents"] = df["Dependents"].replace("3+", "3").astype(int)

# 3. LoanAmount: الوسيط
print("LoanAmount | المتوسط:", df["LoanAmount"].mean().round(2), "| الوسيط:", df["LoanAmount"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

# 4. Loan_Amount_Term: mode
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].mode()[0])

# 5. Credit_History: mode (نسبة موافقة الناقص 0.74 قريبة من مجموعة 1)
df["Credit_History"] = df["Credit_History"].fillna(df["Credit_History"].mode()[0])

# 6. Loan_Status إلى أرقام
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

print(df.isna().sum().sum(), df.shape)

Gender | الناقص: 13 | الأكثر تكرارًا: Male
Married | الناقص: 3 | الأكثر تكرارًا: Yes
Self_Employed | الناقص: 32 | الأكثر تكرارًا: No
Dependents قبل: {'0': 345, '1': 102, '2': 101, '3+': 51, nan: 15}
LoanAmount | المتوسط: 146.41 | الوسيط: 128.0
0 (614, 13)


In [12]:
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)
df["Loan_to_Income"] = (df["LoanAmount"] * 1000) / df["Total_Income"]
df["Monthly_Payment"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]
df["Payment_to_Income"] = df["Monthly_Payment"] / df["Total_Income"]

print(df.shape)
print(df[["Total_Income", "Has_Coapplicant", "Loan_to_Income", "Monthly_Payment", "Payment_to_Income"]].head())

(614, 18)
   Total_Income  Has_Coapplicant  Loan_to_Income  Monthly_Payment  \
0        5849.0                0       21.884083       355.555556   
1        6091.0                1       21.014612       355.555556   
2        3000.0                0       22.000000       183.333333   
3        4941.0                1       24.286582       333.333333   
4        6000.0                0       23.500000       391.666667   

   Payment_to_Income  
0           0.060789  
1           0.058374  
2           0.061111  
3           0.067463  
4           0.065278  


### لماذا عملت هذه الأعمدة؟
البنك يهمه قدرة العميل على السداد وليس الدخل وحده، فولّدت:
- `Total_Income`: دخل المتقدم + دخل الشريك، لأن السداد يعتمد على دخل الأسرة كلها.
- `Has_Coapplicant`: هل يوجد شريك (1 أو 0)، لأن وجوده غيّر نسبة الموافقة (71.8% مقابل 64.8%).
- `Loan_to_Income`: مبلغ القرض بالريال ÷ الدخل الكلي، يقيس حجم القرض مقارنة بالدخل.
- `Monthly_Payment`: مبلغ القرض ÷ مدة السداد بالأشهر (تقريبي بدون فوائد).
- `Payment_to_Income`: القسط الشهري ÷ الدخل الكلي، وهو أقرب مقياس للقدرة على السداد.

## القسم 5fuature selection

In [13]:
# الارتباط مع Target
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term",
            "Credit_History", "Total_Income", "Has_Coapplicant", "Loan_to_Income",
            "Monthly_Payment", "Payment_to_Income"]
print(df[num_cols + ["Loan_Status"]].corr()["Loan_Status"].drop("Loan_Status")
      .sort_values(key=abs, ascending=False).round(3))

# الفجوة في نسبة الموافقة للأعمدة التصنيفية
for col in ["Property_Area", "Dependents", "Education", "Married", "Gender", "Self_Employed"]:
    rates = df.groupby(col)["Loan_Status"].mean()
    print(col, "| الفجوة:", round(rates.max() - rates.min(), 3))

# التكرار بين الدخلين
print("الارتباط ApplicantIncome / Total_Income:", round(df["ApplicantIncome"].corr(df["Total_Income"]), 3))

Credit_History       0.541
Loan_to_Income      -0.086
Has_Coapplicant      0.075
CoapplicantIncome   -0.059
Payment_to_Income   -0.051
LoanAmount          -0.033
Total_Income        -0.031
Loan_Amount_Term    -0.023
Monthly_Payment     -0.012
ApplicantIncome     -0.005
Name: Loan_Status, dtype: float64
Property_Area | الفجوة: 0.154
Dependents | الفجوة: 0.105
Education | الفجوة: 0.096
Married | الفجوة: 0.089
Gender | الفجوة: 0.022
Self_Employed | الفجوة: 0.005
الارتباط ApplicantIncome / Total_Income: 0.893


## القسم 6:حفظ الداتا الجاهزة

In [14]:
selected = ["Credit_History", "Property_Area", "Dependents", "Education", "Married",
            "Has_Coapplicant", "Loan_to_Income", "Payment_to_Income", "Total_Income"]
model_ready = df[selected + ["Loan_Status"]]
print(model_ready.shape)
model_ready.to_csv("model_ready.csv", index=False)

(614, 10)
